# Trained Embedding Model — Ridge Regression on Text Embeddings

Unlike S1 Hard70 (deterministic rules), this model **learns** from embeddings:
1. Load 122K daily embeddings (50 tickers)
2. Align with forward returns
3. Train Ridge regression: embedding → predicted return
4. Cross-sectional tilt → portfolio
5. Compare vs BH and vs deterministic S1

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
%matplotlib inline
import sys, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import TimeSeriesSplit
from src.backtest.engine import decompose_alpha_beta
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, RED, WHITE, CYAN, PURPLE
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

## 1. Load Embeddings + Prices + Returns

In [ ]:
embs = np.load('cache/text/top50_daily_embeddings.npy', mmap_mode='r')
meta = pd.read_csv('cache/text/top50_daily_metadata.csv')
meta['date'] = pd.to_datetime(meta['date'], errors='coerce', utc=True).dt.tz_localize(None).dt.normalize()
meta = meta.dropna(subset=['date'])
embs = embs[meta.index]

cached = pd.read_pickle('cache/data/top50_prepared_data.pkl')
ret_df = cached['ret_df']
known_ret = cached['known_ret']
intensity_df = cached['intensity_df']
regime = cached['regime']
mask = cached['mask']
split = 2694

emb_cols = [f'emb_{i}' for i in range(embs.shape[1])]
emb_df = pd.DataFrame(embs, columns=emb_cols)
emb_df['ticker'] = meta['ticker'].values
emb_df['date'] = meta['date'].values

daily_emb = emb_df.groupby(['ticker', 'date'], as_index=False)[emb_cols].mean()

pd.DataFrame({
    '': ['embeddings', 'tickers', 'dates', 'ret_df', 'train_split', 'test_split'],
    'value': [f'{embs.shape}', daily_emb['ticker'].nunique(), len(daily_emb['date'].unique()),
              f'{ret_df.shape}', split, len(ret_df)-split]
})

## 2. Build Training Dataset — Embedding → Forward Return

In [ ]:
tickers_list = sorted(daily_emb['ticker'].unique())
common_dates = sorted(set(daily_emb['date'].unique()) & set(ret_df.index))
train_dates = common_dates[:split]
test_dates = common_dates[split:]

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

for t in tickers_list:
    emb_t = daily_emb[daily_emb['ticker'] == t].set_index('date')[emb_cols]
    emb_t = emb_t.reindex(common_dates).ffill().bfill()

    if t in ret_df.columns:
        y = ret_df[t].reindex(common_dates)
    else:
        continue

    X_train = emb_t.loc[train_dates].values
    y_train = y.loc[train_dates].values
    X_test = emb_t.loc[test_dates].values
    y_test = y.loc[test_dates].values

    valid_train = ~np.isnan(y_train) & ~np.isnan(X_train).any(axis=1)
    valid_test = ~np.isnan(y_test) & ~np.isnan(X_test).any(axis=1)

    if valid_train.sum() > 50:
        X_train_list.append(X_train[valid_train])
        y_train_list.append(y_train[valid_train])
        X_test_list.append(X_test[valid_test])
        y_test_list.append(y_test[valid_test])

X_train_all = np.vstack(X_train_list)
y_train_all = np.hstack(y_train_list)
X_test_all = np.vstack(X_test_list)
y_test_all = np.hstack(y_test_list)

pd.DataFrame({
    '': ['train_samples', 'test_samples', 'features', 'tickers'],
    'value': [len(y_train_all), len(y_test_all), X_train_all.shape[1], len(X_train_list)]
})

## 3. Train Ridge Regression

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_all)
X_test_s = scaler.transform(X_test_all)

pca = PCA(n_components=32)
X_train_p = pca.fit_transform(X_train_s)
X_test_p = pca.transform(X_test_s)

alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
best_alpha, best_score = 1.0, -np.inf
tscv = TimeSeriesSplit(n_splits=3)

for alpha in alphas:
    scores = []
    for tr_idx, val_idx in tscv.split(X_train_p):
        m = Ridge(alpha=alpha)
        m.fit(X_train_p[tr_idx], y_train_all[tr_idx])
        pred = m.predict(X_train_p[val_idx])
        scores.append(np.corrcoef(pred, y_train_all[val_idx])[0, 1])
    avg = np.mean(scores)
    if avg > best_score:
        best_score = avg
        best_alpha = alpha

model = Ridge(alpha=best_alpha)
model.fit(X_train_p, y_train_all)

train_pred = model.predict(X_train_p)
test_pred = model.predict(X_test_p)

pd.DataFrame({
    '': ['best_alpha', 'pca_dim', 'train_corr', 'test_corr', 'train_r2', 'test_r2'],
    'value': [
        best_alpha, 32,
        f'{np.corrcoef(train_pred, y_train_all)[0,1]:.4f}',
        f'{np.corrcoef(test_pred, y_test_all)[0,1]:.4f}',
        f'{model.score(X_train_p, y_train_all):.4f}',
        f'{model.score(X_test_p, y_test_all):.4f}',
    ]
})

## 4. Generate Signals Per Ticker + Portfolio Construction

In [ ]:
signal_df = pd.DataFrame(0.0, index=common_dates, columns=tickers_list)

for t in tickers_list:
    emb_t = daily_emb[daily_emb['ticker'] == t].set_index('date')[emb_cols]
    emb_t = emb_t.reindex(common_dates).ffill().bfill()
    X = emb_t.values
    X = np.nan_to_num(X, 0.0)
    X_s = scaler.transform(X)
    X_p = pca.transform(X_s)
    preds = model.predict(X_p)
    signals = np.clip(preds, -0.5, 0.5)
    signal_df[t] = signals

mask_aligned = mask.reindex(index=common_dates, columns=tickers_list).fillna(0.0)
ret_df_aligned = ret_df.reindex(index=common_dates, columns=tickers_list).fillna(0.0)
known_ret_aligned = known_ret.reindex(index=common_dates, columns=tickers_list).fillna(0.0)

def build_tilt(raw):
    x = (raw * mask_aligned).fillna(0.0)
    x = x.sub(x.mean(axis=1), axis=0)
    x = x.div(x.abs().sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    return x

tilt = build_tilt(signal_df)
bh_w = mask_aligned.div(mask_aligned.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
bh_ret = (bh_w * ret_df_aligned.fillna(0.0)).sum(axis=1)
mkt_trend = known_ret_aligned.mean(axis=1).rolling(20, min_periods=10).mean().fillna(0.0)
exposure = pd.Series(np.where(mkt_trend > 0.0, 1.30, 1.00), index=common_dates)

w_tilted = (bh_w + 0.05 * tilt).clip(lower=0.0)
w_tilted = w_tilted.div(w_tilted.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
base_ret = (w_tilted * ret_df_aligned.fillna(0.0)).sum(axis=1)
strat_ret = exposure * base_ret

split_dates = common_dates[split]
strat_test = strat_ret.loc[split_dates:]
bh_test_ret = bh_ret.loc[split_dates:]

def stats(r):
    r = r.fillna(0.0); t = (1+r).prod()-1
    a = (1+t)**(252/max(len(r),1))-1; v = r.std(ddof=0)*np.sqrt(252)
    return t, a, v, a/v if v>0 else 0

s_t, s_a, s_v, s_sh = stats(strat_test)
b_t, b_a, b_v, b_sh = stats(bh_test_ret)
ab = decompose_alpha_beta(strat_test, bh_test_ret, 252)

pd.DataFrame({
    '': ['Test Return', 'BH Return', 'Excess', 'Ann. Return', 'BH Ann.', 'Sharpe', 'BH Sharpe', 'Alpha', 'Beta', 'R²'],
    'Value': [
        f'{s_t*100:+.2f}%', f'{b_t*100:+.2f}%', f'{(s_t-b_t)*100:+.2f}%',
        f'{s_a*100:.2f}%', f'{b_a*100:.2f}%',
        f'{s_sh:.3f}', f'{b_sh:.3f}',
        f'{ab["alpha_pct"]:+.2f}%', f'{ab["beta"]:.3f}', f'{ab["r_squared"]:.3f}'
    ]
})

## 5. Equity Curve — Trained Model vs Buy & Hold

In [ ]:
eq_model = (1.0 + strat_ret).cumprod() * 100
eq_bh = (1.0 + bh_ret).cumprod() * 100

images = _root / 'images'
images.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(20, 10))
fig.patch.set_facecolor(BG)
ax.plot(eq_bh.index, eq_bh.values, color=WHITE, lw=1.2, ls='--', alpha=0.5, label=f'Buy & Hold ({b_t*100:+.1f}%)')
ax.plot(eq_model.index, eq_model.values, color=PURPLE, lw=2.5, label=f'Trained Ridge Model ({s_t*100:+.1f}%)')
ax.axvline(split_dates, color=WHITE, ls=':', alpha=0.3, label='Train/Test split')
ax.fill_between(eq_model.index, eq_model.values, 100, where=eq_model.values>=100, color=GREEN, alpha=0.06)
ax.fill_between(eq_model.index, eq_model.values, 100, where=eq_model.values<100, color=RED, alpha=0.06)
ax.axhline(100, color='white', ls='--', alpha=0.15)
ax.set_title(f'Trained Ridge Model on Embeddings vs BH — 50 tickers  |  test={s_t*100:+.1f}% vs BH={b_t*100:+.1f}%  |  alpha={ab["alpha_pct"]:+.1f}%  |  sharpe={s_sh:.2f}',
             color='white', fontsize=12, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')
fig.tight_layout()
fig.savefig(images / 'trained_model_vs_bh.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 6. Feature Importance (Top PCA Components vs Returns)

In [ ]:
coefs = pd.Series(model.coef_, index=[f'PC_{i+1}' for i in range(32)])
top_coefs = coefs.abs().nlargest(10)

fig, ax = plt.subplots(figsize=(20, 7))
fig.patch.set_facecolor(BG)
colors = [GOLD if v > 0 else RED for v in coefs[top_coefs.index]]
ax.barh(range(len(top_coefs)), coefs[top_coefs.index].values, color=colors, alpha=0.85, edgecolor='white', lw=0.3)
ax.set_yticks(range(len(top_coefs)))
ax.set_yticklabels(top_coefs.index, color='white', fontsize=10)
ax.axvline(0, color='white', lw=0.5)
ax.set_title('Top 10 PCA Component Weights (Ridge)', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12, axis='x')
fig.tight_layout()
fig.savefig(images / 'trained_model_importance.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()